# 10 — Otimização temporal do HGB com Optuna

**Implementação preparada; execução real pendente.** Este notebook revisa artefatos salvos; não inicia treinamento nem consulta setembro. Versione o protocolo e a implementação, prepare e execute pelo terminal conforme o [procedimento](../../docs/modeling/EVALUATION_PROTOCOL.md#otimização-temporal-do-hgb-com-optuna-v1).

Após a primeira chamada, registre custo dos três fits de referência e do primeiro trial de três fits. Continuar o estudo não reinicia orçamento; a seleção não autoriza promoção. Confirmação estatística terá protocolo próprio.


In [ ]:
import json
from pathlib import Path

import pandas as pd

from fraud_detection_mlops.config import PROJECT_ROOT
from fraud_detection_mlops.modeling.experiments import (
    hgb_optimization as hgb_optuna,
)
from fraud_detection_mlops.modeling.experiments import (
    temporal_development_data as development,
)

# Substitua pelo development_path retornado por hgb_optuna prepare.
DEVELOPMENT_PATH = Path("CAMINHO_RETORNADO")
if not DEVELOPMENT_PATH.is_absolute():
    DEVELOPMENT_PATH = PROJECT_ROOT / DEVELOPMENT_PATH
STUDY_PATH = DEVELOPMENT_PATH / "study"
result = hgb_optuna.verify_hgb_optimization(STUDY_PATH)
policy = development.load_policy()
result

## Comparação pareada e custos

AP média é o objetivo dos três cortes. O gate considera ganho médio de AP e precisão @100, além da perda em cada corte. `candidate_for_confirmation_review` só pode aparecer após concluir o orçamento; não comprova superioridade.

RSS é o pico acumulado do processo, não memória isolada de cada fit. Confira os tempos antes dos lotes seguintes.


In [ ]:
reference = pd.DataFrame([result["reference"]])
comparison = pd.DataFrame(result["comparison"])
display(reference)
display(comparison)

In [ ]:
rows = []
for receipt in sorted((STUDY_PATH / "models").glob("*/*/result.json")):
    record = json.loads(receipt.read_text())
    rows.append(
        {
            "candidate": receipt.parent.parent.name,
            "fold": record["fold_id"],
            "training_rows": record["training_rows"],
            "validation_rows": record["validation_rows"],
            "average_precision": record["metrics"]["average_precision"],
            "precision_at_100": record["metrics"]["daily_customer_precision_at_100"],
            **record["timings"],
            "mlflow_run_id": record["tracking"]["run_id"],
        }
    )
folds = pd.DataFrame(rows)
folds

## Fila diária e recall

Clientes se repetem entre dias e janelas. A fila retrospectiva usa máximo score por cliente/dia; não é autorização online. Recall descreve a fração de clientes com fraude alcançados em cada dia. Não tratar dispersão desses dias como intervalo de confiança.


In [ ]:
daily_rows = []
for receipt in sorted((STUDY_PATH / "models").glob("*/*/result.json")):
    predictions = pd.read_parquet(receipt.parent / "validation_predictions.parquet")
    predictions["day"] = predictions.TX_DATETIME.dt.strftime("%Y-%m-%d")
    customers = predictions.groupby(["day", "CUSTOMER_ID"]).TX_FRAUD.max()
    fraudulent = customers.groupby(level="day").sum()
    record = json.loads(receipt.read_text())
    for day in record["metrics"]["daily_customer_metrics"]:
        count = int(fraudulent.loc[day["date"]])
        found = day["fraudulent_customers_in_alerts"]
        daily_rows.append(
            {
                "candidate": receipt.parent.parent.name,
                "fold": record["fold_id"],
                **day,
                "fraudulent_customers": count,
                "fraudulent_customers_outside_alerts": count - found,
                "recall_at_100": found / count if count else None,
            }
        )
pd.DataFrame(daily_rows)

## Registrar a decisão

Documentar orçamento utilizado, falhas, efeitos por corte e custo local. Se o estudo estiver incompleto, ainda não há candidato autorizado à revisão confirmatória. Se concluído, avaliar retenção ou candidato elegível sem voltar a maio nem abrir setembro aqui.

Antes da confirmação: definir hipótese, efeito relevante, unidade de inferência e tratamento da dependência temporal/por cliente, conforme o [marco estatístico](../../docs/project/ROADMAP.md#marco-de-análise-estatística-e-testes-de-hipóteses). Não usar esta busca para afirmar degradação do modelo fixo.

Resultado informado pelo autor: 20 trials, nenhuma falha, 63 fits e
`retain_reference`. O [recibo do relatório](../../references/evidence/hgb_optuna_author_report_2026-10-09.json)
registra origem e limites da conferência; esta célula é narrativa, não um output
produzido pela execução deste notebook.
